<a href="https://colab.research.google.com/github/varakalicheva-hub/compling/blob/main/%D0%9A%D0%BE%D0%B4_%D0%B4%D0%BB%D1%8F_%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B8_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%B0_%D0%BF%D0%BE_%D1%82%D0%BE%D1%87%D0%BD%D0%BE%D0%BC%D1%83_%D0%B8_%D1%87%D0%B0%D1%81%D1%82%D0%B8%D1%87%D0%BD%D0%BE%D0%BC%D1%83_%D1%81%D0%BE%D0%B2%D0%BF%D0%B0%D0%B4%D0%B5%D0%BD%D0%B8%D1%8E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# Лемматизация (pymorphy3 / pymorphy2)
# ---------------------------------------------------------------------------

_morph = None
_USE_MORPH = False
# pymorphy3 — актуальная версия с тем же API (pymorphy2 не работает на
# Python 3.11+, т.к. там удалён inspect.getargspec). Пробуем оба.
for _pkg in ('pymorphy3', 'pymorphy2'):
    try:
        _pm = __import__(_pkg)
        _morph = _pm.MorphAnalyzer()
        _USE_MORPH = True
        break
    except Exception:
        continue

if not _USE_MORPH:
    print(
        "[WARNING] pymorphy3/pymorphy2 не установлен. Лемматизация отключена.\n"
        "Установите: pip install pymorphy3 pymorphy3-dicts-ru"
    )

_lemma_cache: dict = {}
_phrase_cache: dict = {}


def lemmatize_word(word: str) -> str:
    if not _USE_MORPH:
        return word
    if word not in _lemma_cache:
        _lemma_cache[word] = _morph.parse(word)[0].normal_form
    return _lemma_cache[word]


def get_lemma_tokens(phrase: str) -> list:
    """Список лемматизированных токенов фразы (с кэшированием)."""
    if phrase not in _phrase_cache:
        _phrase_cache[phrase] = [lemmatize_word(w) for w in phrase.split()]
    return _phrase_cache[phrase]


# ---------------------------------------------------------------------------
# Нормализация текста
# ---------------------------------------------------------------------------

def normalize_text(text) -> str:
    """Приводит к нижнему регистру, сворачивает пробелы. Заодно лечит дубли
    меток вида 'Материалы курса' / 'материалы курса' в датасете."""
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', str(text).strip().lower())


# ---------------------------------------------------------------------------
# Нормализация метки тональности
# ---------------------------------------------------------------------------
# В данных встречаются грязные значения: 'нега' (обрезанное), NaN.
# Приводим всё к канонической форме, иначе квадруплеты с такими метками
# никогда не совпадут по sentiment.

_SENTIMENT_MAP = {
    'положительная': 'положительная',
    'позитивная': 'положительная',
    'поз': 'положительная',
    'pos': 'положительная',
    'positive': 'положительная',
    'отрицательная': 'отрицательная',
    'негативная': 'отрицательная',
    'нега': 'отрицательная',
    'нег': 'отрицательная',
    'neg': 'отрицательная',
    'negative': 'отрицательная',
    'нейтральная': 'нейтральная',
    'neu': 'нейтральная',
    'neutral': 'нейтральная',
}


def normalize_sentiment(value) -> str:
    """Канонизация метки тональности с обработкой опечаток и обрезанных слов."""
    v = normalize_text(value)
    if v in _SENTIMENT_MAP:
        return _SENTIMENT_MAP[v]
    # частичное совпадение по началу слова ('нега' -> 'негативная')
    for key, canon in _SENTIMENT_MAP.items():
        if v and (v.startswith(key) or key.startswith(v)):
            return canon
    return v  # неизвестное значение оставляем как есть (попадёт в FP/FN)


# ---------------------------------------------------------------------------
# Стоп-слова (адаптация списка из Hua et al. 2025, Sec. 4.1.4, под русский)
# ---------------------------------------------------------------------------

STOPWORDS = {
    'и', 'в', 'во', 'не', 'на', 'с', 'со', 'а', 'но', 'что', 'это', 'этот',
    'эта', 'эти', 'тот', 'та', 'те', 'то', 'как', 'бы', 'же', 'из', 'у', 'к',
    'о', 'об', 'для', 'по', 'за', 'до', 'очень', 'весьма', 'крайне', 'супер',
    'абсолютно', 'совершенно', 'действительно', 'реально', 'есть', 'был',
    'была', 'было', 'были', 'быть',
}


def remove_stopwords(tokens: list) -> list:
    """Удаляет стоп-слова. Если после удаления список пуст — возвращает
    исходный (чтобы короткая фраза вроде 'очень' не превратилась в пустую)."""
    filtered = [t for t in tokens if t not in STOPWORDS]
    return filtered if filtered else tokens


# ---------------------------------------------------------------------------
# Rouge-L F1 (ядро FTS-скоринга)
# ---------------------------------------------------------------------------

def _lcs_length(a: list, b: list) -> int:
    """Длина наибольшей общей подпоследовательности двух списков токенов."""
    if not a or not b:
        return 0
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[len(a)][len(b)]


def fts_score(pred: str, gold: str) -> float:
    """Flexible Text Similarity score in [0, 1].

    Адаптация FTS из Hua et al. 2025: Rouge-L F1 между лемматизированными
    токенами gold и pred после удаления стоп-слов.

    Отличие от статьи: токены лемматизируются (pymorphy), т.к. для русского
    языка сравнение по словоформам занижает совпадение из-за богатой
    морфологии (напр. 'хороший преподаватель' vs 'хорошего преподавателя').
    """
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)

    if not pred_n and not gold_n:
        return 1.0
    if not pred_n or not gold_n:
        return 0.0

    g_tokens = remove_stopwords(get_lemma_tokens(gold_n))
    p_tokens = remove_stopwords(get_lemma_tokens(pred_n))

    if not g_tokens or not p_tokens:
        return 0.0

    lcs = _lcs_length(p_tokens, g_tokens)
    if lcs == 0:
        return 0.0

    precision = lcs / len(p_tokens)
    recall = lcs / len(g_tokens)
    return 2 * precision * recall / (precision + recall)


def length_threshold(gold: str) -> float:
    """Динамический порог T по длине gold (Hua et al. 2025):
    T = 0.5 для |g| in [0,2], 0.6 для [3,4], 0.7 для |g| >= 5.
    Длина — по лемматизированным токенам после удаления стоп-слов."""
    g_tokens = remove_stopwords(get_lemma_tokens(normalize_text(gold)))
    n = len(g_tokens)
    if n <= 2:
        return 0.5
    elif n <= 4:
        return 0.6
    else:
        return 0.7


# ---------------------------------------------------------------------------
# Partial match для текстовых компонентов (FTS + порог)
# ---------------------------------------------------------------------------

def is_partial_match(pred: str, gold: str) -> bool:
    return fts_score(pred, gold) >= length_threshold(gold)


def partial_quad_match(pred_quad, gold_quad) -> bool:
    """Квадруплет совпадает (partial): category и sentiment — строго,
    aspect term и opinion — по FTS-порогу."""
    pred_term, pred_op, pred_asp, pred_sent = pred_quad
    gold_term, gold_op, gold_asp, gold_sent = gold_quad

    if pred_asp != gold_asp or pred_sent != gold_sent:
        return False

    return (is_partial_match(pred_term, gold_term)
            and is_partial_match(pred_op, gold_op))


# ---------------------------------------------------------------------------
# Unit-similarity для матрицы OBP (непрерывный балл)
# ---------------------------------------------------------------------------

def quad_similarity(pred_quad, gold_quad) -> float:
    """Взвешенная (равными весами) сумма покомпонентных баллов пары
    квадруплетов — элемент матрицы D в OBP.

    Категория в датасете одноуровневая, поэтому частичный балл за
    совпадение главной категории (как в EduRABSA с метками 'Main - sub')
    здесь не применяется: asp_s принимает только 1 или 0."""
    pred_term, pred_op, pred_asp, pred_sent = pred_quad
    gold_term, gold_op, gold_asp, gold_sent = gold_quad

    term_s = fts_score(pred_term, gold_term)
    op_s = fts_score(pred_op, gold_op)
    asp_s = 1.0 if pred_asp == gold_asp else 0.0
    sent_s = 1.0 if pred_sent == gold_sent else 0.0

    return (term_s + op_s + asp_s + sent_s) / 4


# ---------------------------------------------------------------------------
# Build quads
# ---------------------------------------------------------------------------

def build_quad_list(rows: pd.DataFrame) -> list:
    """Возвращает список кортежей (term, opinion, aspect, sentiment).
    sentiment канонизируется, остальные компоненты нормализуются."""
    quads = []
    for _, row in rows.iterrows():
        term = normalize_text(row['aspect_term'])
        op = normalize_text(row['opinion'])
        asp = normalize_text(row['aspect'])
        sent = normalize_sentiment(row['sentiment'])
        quads.append((term, op, asp, sent))
    return quads


# ---------------------------------------------------------------------------
# OPTIMAL MATCHING (OBP)
# ---------------------------------------------------------------------------

def optimal_match(pred_list, gold_list, match_fn):
    """Булев вариант: матрица 0/1, паросочетание максимизирует число
    совпадений. Используется для exact-режима и покомпонентных метрик."""
    n_pred, n_gold = len(pred_list), len(gold_list)
    if n_pred == 0 and n_gold == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gold
    if n_gold == 0:
        return 0, n_pred, 0

    match_matrix = np.zeros((n_pred, n_gold), dtype=int)
    for i, pred in enumerate(pred_list):
        for j, gold in enumerate(gold_list):
            if match_fn(pred, gold):
                match_matrix[i, j] = 1

    row_ind, col_ind = linear_sum_assignment(-match_matrix)
    tp = sum(match_matrix[i, j] for i, j in zip(row_ind, col_ind))
    return tp, n_pred - tp, n_gold - tp


def optimal_match_weighted(pred_list, gold_list, sim_fn, match_fn):
    """FTS-OBP вариант: паросочетание выбирается по НЕПРЕРЫВНОЙ матрице
    сходства (sim_fn), затем выбранная пара проверяется на match (match_fn).
    Точнее соответствует статье, чем булева матрица."""
    n_pred, n_gold = len(pred_list), len(gold_list)
    if n_pred == 0 and n_gold == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gold
    if n_gold == 0:
        return 0, n_pred, 0

    sim_matrix = np.zeros((n_pred, n_gold), dtype=float)
    for i, pred in enumerate(pred_list):
        for j, gold in enumerate(gold_list):
            sim_matrix[i, j] = sim_fn(pred, gold)

    row_ind, col_ind = linear_sum_assignment(-sim_matrix)
    tp = sum(1 for i, j in zip(row_ind, col_ind)
             if match_fn(pred_list[i], gold_list[j]))
    return tp, n_pred - tp, n_gold - tp


# ---------------------------------------------------------------------------
# PRF
# ---------------------------------------------------------------------------

def compute_prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


# ---------------------------------------------------------------------------
# Подготовка датафрейма
# ---------------------------------------------------------------------------

def prepare_df(df: pd.DataFrame, name: str = "") -> pd.DataFrame:
    """Чистит датафрейм: убирает строки с пустым или пробельным review_id,
    приводит типы. Сообщает, сколько строк отброшено."""
    df = df.copy()
    n_before = len(df)
    rid = df['review_id'].astype(str).str.strip()
    df = df[df['review_id'].notna() & (rid != '') & (rid.str.lower() != 'nan')]
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"[{name}] отброшено строк с пустым review_id: {n_dropped}")
    df['review_id'] = df['review_id'].astype(str).str.strip()
    return df


# ---------------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------------

def compute_metrics(golden_df: pd.DataFrame, pred_df: pd.DataFrame,
                    mode: str = 'exact') -> dict:
    """Считает P/R/F1 по компонентам (aspect term, opinion, aspect,
    sentiment) и по квадруплету в целом.

    mode='exact'   — строгое совпадение всех компонентов.
    mode='partial' — гибкое сопоставление текстовых компонентов через FTS
                     (Rouge-L по леммам + порог по длине); категория и
                     тональность по-прежнему сравниваются строго.
    """
    golden_df = prepare_df(golden_df, "golden")
    pred_df = prepare_df(pred_df, "pred")

    golden_grouped = golden_df.groupby('review_id')
    pred_grouped = pred_df.groupby('review_id')

    counters = {k: [0, 0, 0] for k in ['quad', 'term', 'op', 'asp', 'sent']}

    def add(key, tp=0, fp=0, fn=0):
        counters[key][0] += tp
        counters[key][1] += fp
        counters[key][2] += fn

    all_review_ids = set(golden_grouped.groups) | set(pred_grouped.groups)

    for rid in all_review_ids:
        gold_rows = (golden_grouped.get_group(rid)
                     if rid in golden_grouped.groups else pd.DataFrame())
        pred_rows = (pred_grouped.get_group(rid)
                     if rid in pred_grouped.groups else pd.DataFrame())

        gold_list = build_quad_list(gold_rows)
        pred_list = build_quad_list(pred_rows)

        if mode == 'exact':
            tp, fp, fn = optimal_match(pred_list, gold_list, lambda p, g: p == g)
            add('quad', tp=tp, fp=fp, fn=fn)

            for key, idx in [('term', 0), ('op', 1), ('asp', 2), ('sent', 3)]:
                g_vals = [q[idx] for q in gold_list]
                p_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match(p_vals, g_vals, lambda p, g: p == g)
                add(key, tp=tp, fp=fp, fn=fn)

        elif mode == 'partial':
            tp, fp, fn = optimal_match_weighted(
                pred_list, gold_list,
                sim_fn=quad_similarity,
                match_fn=partial_quad_match,
            )
            add('quad', tp=tp, fp=fp, fn=fn)

            for key, idx in [('term', 0), ('op', 1)]:
                g_vals = [q[idx] for q in gold_list]
                p_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match_weighted(
                    p_vals, g_vals,
                    sim_fn=fts_score,
                    match_fn=is_partial_match,
                )
                add(key, tp=tp, fp=fp, fn=fn)

            for key, idx in [('asp', 2), ('sent', 3)]:
                g_vals = [q[idx] for q in gold_list]
                p_vals = [q[idx] for q in pred_list]
                tp, fp, fn = optimal_match(p_vals, g_vals, lambda p, g: p == g)
                add(key, tp=tp, fp=fp, fn=fn)

        else:
            raise ValueError(f"mode должен быть 'exact' или 'partial', получено: {mode}")

    return {
        'Aspect Term': compute_prf(*counters['term']),
        'Opinion':     compute_prf(*counters['op']),
        'Aspect':      compute_prf(*counters['asp']),
        'Sentiment':   compute_prf(*counters['sent']),
        'Quad':        compute_prf(*counters['quad']),
    }


# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------

if __name__ == "__main__":

    golden_df = pd.read_csv('golden_dataset.csv')
    pred_df = pd.read_csv('predictions_dataset_gpt.csv')

    print("\n=== Exact Match ===")
    for name, (p, r, f) in compute_metrics(golden_df, pred_df, mode='exact').items():
        print(f"{name:<15} P={p:.4f} R={r:.4f} F1={f:.4f}")

    print("\n=== Partial Match (FTS-style) ===")
    for name, (p, r, f) in compute_metrics(golden_df, pred_df, mode='partial').items():
        print(f"{name:<15} P={p:.4f} R={r:.4f} F1={f:.4f}")